# Agent2Agent Discovery And Invocation Localhost Demo

This notebook demonstrates the local Agent2Agent (A2A) flow in InnomightLabs:

1. Read the public facilitator Agent Card from `/.well-known/agent-card.json`.
2. Inspect the enabled agents advertised by the facilitator.
3. Fetch the auxiliary `/a2a/agents` listing.
4. Fetch an agent-scoped card for a discovered agent.
5. Invoke the agent with `message:send` using an agent API key.
6. Fetch persisted A2A task state.
7. Exercise `message:stream` for Server-Sent Events.


## Prerequisites

Start the API locally before running the notebook. For the current local deploy setup, the public API is expected at:

```text
http://localhost:1455
```

In the SPA, open an agent, create an API key if needed, and enable `Agent2Agent Discovery` on the agent overview page.

Do not commit live API keys into notebooks. Set the API key as an environment variable before launching Jupyter:

```bash
export A2A_API_KEY='pk_live_...'
```

You can override the API base URL too:

```bash
export A2A_API_BASE_URL='http://localhost:1455'
```


In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

API_BASE_URL = os.getenv("A2A_API_BASE_URL", "http://localhost:1455").rstrip("/")
A2A_API_KEY = os.getenv("A2A_API_KEY")

print("API_BASE_URL:", API_BASE_URL)
print("A2A_API_KEY configured:", bool(A2A_API_KEY))


API_BASE_URL: http://localhost:1455
A2A_API_KEY configured: True


## Helper Functions

These helpers use Python's standard library so the notebook does not require extra dependencies.


In [15]:
def request_json(path: str, *, method: str = "GET", payload: dict | None = None, headers: dict[str, str] | None = None) -> dict:
    url = f"{API_BASE_URL}{path}"
    body = None
    request_headers = dict(headers or {})
    if payload is not None:
        body = json.dumps(payload).encode("utf-8")
        request_headers.setdefault("Content-Type", "application/a2a+json")
    request = Request(url, data=body, headers=request_headers, method=method)
    try:
        with urlopen(request, timeout=120) as response:
            raw = response.read().decode("utf-8")
            return json.loads(raw) if raw else {}
    except HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{method} {url} failed with HTTP {exc.code}: {raw}") from exc
    except URLError as exc:
        raise RuntimeError(f"{method} {url} failed: {exc}") from exc


def request_sse(path: str, *, payload: dict, headers: dict[str, str]) -> list[dict]:
    url = f"{API_BASE_URL}{path}"
    request = Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={**headers, "Content-Type": "application/a2a+json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=120) as response:
            raw = response.read().decode("utf-8")
    except HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"POST {url} failed with HTTP {exc.code}: {raw}") from exc

    events = []
    for block in raw.strip().split("\n\n"):
        if not block.startswith("data: "):
            continue
        events.append(json.loads(block.removeprefix("data: ")))
    return events


def print_json(value: object) -> None:
    print(json.dumps(value, indent=2, sort_keys=False))


## Step 1: Fetch The Well-Known Facilitator Agent Card

External A2A clients start here. This card represents the InnomightLabs A2A facilitator for this API host.


In [16]:
facilitator_card = request_json("/.well-known/agent-card.json")
print_json(facilitator_card)


{
  "protocolVersion": "1.0.0",
  "name": "InnomightLabs A2A Facilitator",
  "description": "Discovery entrypoint for InnomightLabs agents enabled for Agent2Agent communication.",
  "url": "http://localhost:1455/a2a",
  "provider": {
    "organization": "InnomightLabs",
    "url": "https://innomightlabs.com"
  },
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": true,
    "extendedAgentCard": false
  },
  "securitySchemes": {
    "agentApiKey": {
      "type": "apiKey",
      "in": "header",
      "name": "Authorization",
      "description": "Use Authorization: Bearer <agent API key>."
    }
  },
  "security": [
    {
      "agentApiKey": []
    }
  ],
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "discover_public_agents",
      "name": "Discover Public Agents",
      "description": "List InnomightLabs agents enabled for Agent2Ag

## Step 2: Inspect Facilitator Metadata

Enabled agents are advertised under `metadata.agents`. Each item is built from the existing Agent row: `agent_id`, `agent_name`, `agent_description`, and the service URL.


In [17]:
metadata = facilitator_card.get("metadata", {})
agents = metadata.get("agents", [])

print("agentsUrl:", metadata.get("agentsUrl"))
print("enabled agent count:", len(agents))
print_json(agents)


agentsUrl: http://localhost:1455/a2a/agents
enabled agent count: 1
[
  {
    "agent_id": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
    "name": "League of Legends Insights Report Generator",
    "description": "An HTML Reports generator for League of Legends",
    "service_url": "http://localhost:1455/a2a/agents/46b64d50-9bc7-4733-bf58-eaf9813cd3fd"
  }
]


## Step 3: Fetch The Auxiliary Agent Listing

`/a2a/agents` returns the same discovery data in a paginated API shape. This is useful for developer tools and manual inspection.


In [18]:
agent_listing = request_json("/a2a/agents?limit=20")
print_json(agent_listing)


{
  "items": [
    {
      "agent_id": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
      "name": "League of Legends Insights Report Generator",
      "description": "An HTML Reports generator for League of Legends",
      "service_url": "http://localhost:1455/a2a/agents/46b64d50-9bc7-4733-bf58-eaf9813cd3fd"
    }
  ],
  "next_cursor": null
}


## Step 4: Choose A Discovered Agent

Pick the first enabled agent from the listing. If this cell fails, enable Agent2Agent Discovery for at least one agent in the SPA.


In [19]:
items = agent_listing.get("items", [])
if not items:
    raise RuntimeError("No A2A-enabled agents found. Enable Agent2Agent Discovery for an agent first.")

selected_agent = items[0]
selected_agent_id = selected_agent["agent_id"]

print("selected_agent_id:", selected_agent_id)
print_json(selected_agent)


selected_agent_id: 46b64d50-9bc7-4733-bf58-eaf9813cd3fd
{
  "agent_id": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
  "name": "League of Legends Insights Report Generator",
  "description": "An HTML Reports generator for League of Legends",
  "service_url": "http://localhost:1455/a2a/agents/46b64d50-9bc7-4733-bf58-eaf9813cd3fd"
}


## Step 5: Fetch The Agent-Scoped Card

This endpoint is an auxiliary card for a specific enabled agent after discovery has already happened through the facilitator.


In [20]:
agent_card = request_json(f"/a2a/agents/{selected_agent_id}/agent-card")
print_json(agent_card)


{
  "protocolVersion": "1.0.0",
  "name": "League of Legends Insights Report Generator",
  "description": "An HTML Reports generator for League of Legends",
  "url": "http://localhost:1455/a2a/agents/46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
  "provider": {
    "organization": "InnomightLabs",
    "url": "https://innomightlabs.com"
  },
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": true,
    "extendedAgentCard": false
  },
  "securitySchemes": {
    "agentApiKey": {
      "type": "apiKey",
      "in": "header",
      "name": "Authorization",
      "description": "Use Authorization: Bearer <agent API key>."
    }
  },
  "security": [
    {
      "agentApiKey": []
    }
  ],
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "chat",
      "name": "Chat With Agent",
      "description": "Send a task or question to this agent.",
      "ta

## Step 6: Validate Public Data Is Sanitized

The public card should expose the agent name, description, A2A URL, protocol metadata, and auth scheme. It should not expose the agent persona, provider credentials, OAuth tokens, or installed skill secrets.


In [21]:
public_payload = json.dumps(agent_card)

for forbidden in ["agent_persona", "agent_provider_api_key", "encrypted_credentials", "oauth", "secret"]:
    print(f"{forbidden!r} present:", forbidden in public_payload)


'agent_persona' present: False
'agent_provider_api_key' present: False
'encrypted_credentials' present: False
'oauth' present: False
'secret' present: False


## Step 7: Prepare Authentication

A2A invocation uses the agent API key as a bearer credential. This notebook intentionally does not store a live key in source control. Set `A2A_API_KEY` in your shell before running the invocation cells.


In [22]:
if not A2A_API_KEY:
    raise RuntimeError("A2A_API_KEY is not configured. Set it before testing invocation.")

auth_headers = {
    "Authorization": f"Bearer {A2A_API_KEY}",
    "Content-Type": "application/a2a+json",
}
redacted = f"{A2A_API_KEY[:12]}...{A2A_API_KEY[-4:]}"
print("Prepared Authorization header for API key:", redacted)
print_json({"Authorization": "Bearer <redacted>", "Content-Type": auth_headers["Content-Type"]})


Prepared Authorization header for API key: pk_live_28ee...76ec
{
  "Authorization": "Bearer <redacted>",
  "Content-Type": "application/a2a+json"
}


## Step 8: Send A Message

`message:send` runs the same internal agent architecture used by the dashboard and returns a completed A2A task. If `contextId` is provided, future requests with the same context and API key reuse the same internal conversation.


In [24]:
context_id = "localhost-demo-context"
message_send_path = f"/a2a/agents/{selected_agent_id}/message:send"
message_send_payload = {
    "message": {
        "messageId": "demo-message-001",
        "role": "ROLE_USER",
        "contextId": context_id,
        "parts": [{"text": "Hello from an A2A client. Please reply with a short acknowledgement."}],
    },
    "configuration": {
        "acceptedOutputModes": ["text/plain"],
    },
}

send_response = request_json(message_send_path, method="POST", payload=message_send_payload, headers=auth_headers)
print_json(send_response)


{
  "task": {
    "id": "e08fe74e-881d-4331-91cb-a5f869e1bd6b",
    "contextId": "localhost-demo-context",
    "agentId": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
    "ownerEmail": "varunshrivastava007@gmail.com",
    "clientKeyId": "cdf12562-bd45-483e-b321-2b774e9a8873",
    "conversationId": "a2a-46b64d50-9bc7-4733-bf58-eaf9813cd3fd-1050b0c84e47e541",
    "status": {
      "state": "TASK_STATE_COMPLETED",
      "message": {
        "messageId": "edfd966c-5289-46f1-a73a-7a47c44e03f7",
        "role": "ROLE_AGENT",
        "parts": [
          {
            "kind": "text",
            "text": "Acknowledged. Ready to assist."
          }
        ]
      }
    },
    "history": [
      {
        "messageId": "demo-message-001",
        "role": "ROLE_USER",
        "parts": [
          {
            "kind": "text",
            "text": "Hello from an A2A client. Please reply with a short acknowledgement."
          }
        ],
        "contextId": "localhost-demo-context"
      }
    ],
  

## Step 9: Fetch The Persisted Task

The returned task ID can be looked up later with the same agent API key. Task lookup is scoped to the key that created the task.


In [25]:
task_id = send_response["task"]["id"]
task_response = request_json(f"/a2a/agents/{selected_agent_id}/tasks/{task_id}", headers=auth_headers)
print_json(task_response)


{
  "task": {
    "id": "e08fe74e-881d-4331-91cb-a5f869e1bd6b",
    "contextId": "localhost-demo-context",
    "agentId": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
    "ownerEmail": "varunshrivastava007@gmail.com",
    "clientKeyId": "cdf12562-bd45-483e-b321-2b774e9a8873",
    "conversationId": "a2a-46b64d50-9bc7-4733-bf58-eaf9813cd3fd-1050b0c84e47e541",
    "status": {
      "state": "TASK_STATE_COMPLETED",
      "message": {
        "messageId": "edfd966c-5289-46f1-a73a-7a47c44e03f7",
        "role": "ROLE_AGENT",
        "parts": [
          {
            "kind": "text",
            "text": "Acknowledged. Ready to assist."
          }
        ],
        "taskId": null,
        "contextId": null
      }
    },
    "history": [
      {
        "messageId": "demo-message-001",
        "role": "ROLE_USER",
        "parts": [
          {
            "kind": "text",
            "text": "Hello from an A2A client. Please reply with a short acknowledgement."
          }
        ],
        "tas

## Step 10: List Tasks For This API Key

This lists tasks for the selected agent and the authenticated API key.


In [26]:
tasks_response = request_json(f"/a2a/agents/{selected_agent_id}/tasks", headers=auth_headers)
print_json(tasks_response)


{
  "items": [
    {
      "id": "e08fe74e-881d-4331-91cb-a5f869e1bd6b",
      "contextId": "localhost-demo-context",
      "agentId": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
      "ownerEmail": "varunshrivastava007@gmail.com",
      "clientKeyId": "cdf12562-bd45-483e-b321-2b774e9a8873",
      "conversationId": "a2a-46b64d50-9bc7-4733-bf58-eaf9813cd3fd-1050b0c84e47e541",
      "status": {
        "state": "TASK_STATE_COMPLETED",
        "message": {
          "messageId": "edfd966c-5289-46f1-a73a-7a47c44e03f7",
          "role": "ROLE_AGENT",
          "parts": [
            {
              "kind": "text",
              "text": "Acknowledged. Ready to assist."
            }
          ],
          "taskId": null,
          "contextId": null
        }
      },
      "history": [
        {
          "messageId": "demo-message-001",
          "role": "ROLE_USER",
          "parts": [
            {
              "kind": "text",
              "text": "Hello from an A2A client. Please reply w

## Step 11: Stream A Message

`message:stream` returns Server-Sent Events. Each `data:` line contains an A2A task status update event.


In [27]:
message_stream_path = f"/a2a/agents/{selected_agent_id}/message:stream"
message_stream_payload = {
    "message": {
        "messageId": "demo-message-002",
        "role": "ROLE_USER",
        "contextId": context_id,
        "parts": [{"text": "Send a second short acknowledgement as a streamed response."}],
    },
    "configuration": {
        "acceptedOutputModes": ["text/plain"],
    },
}

stream_events = request_sse(message_stream_path, payload=message_stream_payload, headers=auth_headers)
print_json(stream_events)


[
  {
    "taskId": "aac7e5d1-fa85-474c-bfb2-86d35e32ca55",
    "contextId": "localhost-demo-context",
    "status": {
      "state": "TASK_STATE_WORKING"
    },
    "final": false
  },
  {
    "taskId": "aac7e5d1-fa85-474c-bfb2-86d35e32ca55",
    "contextId": "localhost-demo-context",
    "status": {
      "state": "TASK_STATE_COMPLETED",
      "message": {
        "messageId": "c5866241-e837-46ec-a234-0dbbc5566fca",
        "role": "ROLE_AGENT",
        "parts": [
          {
            "kind": "text",
            "text": "Acknowledged \u2014 streaming-ready."
          }
        ]
      }
    },
    "final": true
  }
]


## Step 12: Send A Fresh Live Request And Print The Agent Response

Run this after discovery and authentication are configured. It sends a new `message:send` request to the selected A2A agent, prints the full task payload, and extracts the returned agent text for quick inspection.


In [30]:
from uuid import uuid4

live_context_id = f"localhost-demo-{uuid4()}"
live_message_id = f"demo-message-{uuid4()}"
live_payload = {
    "message": {
        "messageId": live_message_id,
        "role": "ROLE_USER",
        "contextId": live_context_id,
        "parts": [
            {
                "text": "Hello from the Agent2Agent localhost notebook. Reply with one concise sentence confirming that A2A invocation is working."
            }
        ],
    },
    "configuration": {
        "acceptedOutputModes": ["text/plain"],
    },
}

live_response = request_json(
    f"/a2a/agents/{selected_agent_id}/message:send",
    method="POST",
    payload=live_payload,
    headers=auth_headers,
)

print_json(live_response)

live_task = live_response["task"]
live_agent_message = live_task.get("status", {}).get("message", {})
live_agent_text = "".join(part.get("text", "") for part in live_agent_message.get("parts", []))

print("Agent response text:")
print(live_agent_text)
print("Task ID:", live_task["id"])
print("Context ID:", live_task["contextId"])
print("Conversation ID:", live_task["conversationId"])


{
  "task": {
    "id": "511c80db-b14d-46dc-970d-ad3d25cf8718",
    "contextId": "localhost-demo-743b3b8d-d9a2-41c9-aa29-487bd1ba0b76",
    "agentId": "46b64d50-9bc7-4733-bf58-eaf9813cd3fd",
    "ownerEmail": "varunshrivastava007@gmail.com",
    "clientKeyId": "cdf12562-bd45-483e-b321-2b774e9a8873",
    "conversationId": "a2a-46b64d50-9bc7-4733-bf58-eaf9813cd3fd-107d2165e1861424",
    "status": {
      "state": "TASK_STATE_FAILED",
      "message": {
        "messageId": "a124cb10-5013-4ebc-9022-abee0b9ab6c1",
        "role": "ROLE_AGENT",
        "parts": [
          {
            "kind": "text",
            "text": "OpenAI stream error (request_id=None): {'type': 'service_unavailable_error', 'code': 'server_is_overloaded', 'message': 'Our servers are currently overloaded. Please try again later.', 'param': None}"
          }
        ]
      }
    },
    "history": [
      {
        "messageId": "demo-message-a23892b9-29fb-420c-8e36-8c9e8db39768",
        "role": "ROLE_USER",
        

## Step 13: Verify The Fresh Task Can Be Retrieved

Use the task ID from Step 12 to verify persisted A2A task state. This should return the same task when called with the same agent API key.


In [ ]:
live_task_lookup = request_json(
    f"/a2a/agents/{selected_agent_id}/tasks/{live_task['id']}",
    headers=auth_headers,
)
print_json(live_task_lookup)


## Expected Outcome

- `/.well-known/agent-card.json` returns the facilitator card.
- `metadata.agents` includes only agents where `is_agent2agent_enabled = true`.
- `/a2a/agents` returns the enabled agent summaries.
- `/a2a/agents/{agent_id}/agent-card` returns a sanitized card for the selected agent.
- `message:send` returns a completed or failed A2A task.
- `tasks/{task_id}` returns the persisted task for the same API key.
- `message:stream` emits A2A task status updates over SSE.
